## Algorithm:  BERT-based Transformer for classifying Fake News


Dataset: [LIAR Dataset]

https://huggingface.co/datasets/liar

Ref Dataset Paper :https://arxiv.org/abs/1705.00648


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore", message="The secret `HF_TOKEN` does not exist in your Colab secrets")

# 1: Load the dataset

The LIAR dataset is a benchmark dataset for detecting fabricated news. It contains 12,836 labeled short political statements compiled from Politifact, a fact-checking web site. All statements are annotated into one of six levels of truthfulness:

"pants-fire" (completely false)

"false"

"barely-true"

"half-true"

"mostly-true"

"true"


In [ ]:
print("Loading the LIAR dataset...")
dataset = load_dataset("liar", trust_remote_code=True)
dataset = dataset.rename_column("label", "labels")

Loading the LIAR dataset...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print("\nFirst 5 samples from the train dataset:")
for i in range(5):
    print(dataset["train"][i])


First 5 samples from the train dataset:
{'id': '2635.json', 'labels': 0, 'statement': 'Says the Annies List political group supports third-trimester abortions on demand.', 'subject': 'abortion', 'speaker': 'dwayne-bohac', 'job_title': 'State representative', 'state_info': 'Texas', 'party_affiliation': 'republican', 'barely_true_counts': 0.0, 'false_counts': 1.0, 'half_true_counts': 0.0, 'mostly_true_counts': 0.0, 'pants_on_fire_counts': 0.0, 'context': 'a mailer'}
{'id': '10540.json', 'labels': 1, 'statement': 'When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.', 'subject': 'energy,history,job-accomplishments', 'speaker': 'scott-surovell', 'job_title': 'State delegate', 'state_info': 'Virginia', 'party_affiliation': 'democrat', 'barely_true_counts': 0.0, 'false_counts': 0.0, 'half_true_counts': 1.0, 'mostly_true_counts': 1.0, 'pants_on_fire_counts': 0.0, 'context': 'a floor speech.'}
{'id': '324

In [ ]:
print("\nColumns in the dataset:", dataset["train"].features)


Columns in the dataset: {'id': Value(dtype='string', id=None), 'labels': ClassLabel(names=['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire'], id=None), 'statement': Value(dtype='string', id=None), 'subject': Value(dtype='string', id=None), 'speaker': Value(dtype='string', id=None), 'job_title': Value(dtype='string', id=None), 'state_info': Value(dtype='string', id=None), 'party_affiliation': Value(dtype='string', id=None), 'barely_true_counts': Value(dtype='float32', id=None), 'false_counts': Value(dtype='float32', id=None), 'half_true_counts': Value(dtype='float32', id=None), 'mostly_true_counts': Value(dtype='float32', id=None), 'pants_on_fire_counts': Value(dtype='float32', id=None), 'context': Value(dtype='string', id=None)}


#  2: Load a pretrained BERT tokenizer

In [ ]:
print("Loading the BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Loading the BERT tokenizer...


#  3: Tokenize the text

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["statement"], padding="max_length", truncation=True)

print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Tokenizing dataset...


Map:   0%|          | 0/1284 [00:00<?, ? examples/s]


#  4: Define an evaluation function

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted",zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA is available. Cache cleared.")
else:
    print("CUDA is not available. Using CPU.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


CUDA is available. Cache cleared.
Using device: cuda


#  5: Set training arguments

In [ ]:
#  6: Set training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="fake_news_bert_experiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,  # Keeps the best version of the model
    metric_for_best_model="f1",
    report_to='none'
)

#  6: Load BERT model (for **6-class classification**)

In [ ]:
print("Loading the BERT model...")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6)

Loading the BERT model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#  7: Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

# 8. Training and Evalution

In [ ]:
#  Train the model
print("Starting training...")
trainer.train()

#   Evaluate the model
print("Evaluating the model...")
trainer.evaluate()


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.688600,1.734413,0.227414,0.233641,0.227414,0.165735
2,1.692500,1.677770,0.260125,0.284365,0.260125,0.248070
3,1.267900,1.938978,0.256231,0.268356,0.256231,0.244415
4,0.981600,2.147719,0.257009,0.272094,0.257009,0.251995
5,0.449200,2.863543,0.266355,0.279671,0.266355,0.264648
6,0.207400,3.663834,0.262461,0.276489,0.262461,0.261633
7,0.190900,4.267904,0.273364,0.280027,0.273364,0.269535
8,0.027500,4.549872,0.261682,0.268039,0.261682,0.259855


Evaluating the model...


{'eval_loss': 4.267903804779053,
 'eval_accuracy': 0.2733644859813084,
 'eval_precision': 0.28002744220406656,
 'eval_recall': 0.2733644859813084,
 'eval_f1': 0.2695353538269611,
 'eval_runtime': 40.6018,
 'eval_samples_per_second': 31.624,
 'eval_steps_per_second': 1.995,
 'epoch': 8.0}

##### Observations:
  - There is potential for improvement by adding external knowledge sources or using larger transformer models.


##### ***Conclusion***:
  - Fine-tuned BERT model on LIAR dataset.
-  Carried out preprocessing through tokenization and encoding of the labels.
-   Evaluating the model's performance with a set of evaluation metrics.
- Observed where the model had to get better, such as support for uncertain sentences and data augmentation.

